# Evidence-per-Token, Tier A.5: does Gemma 4 use the evidence?

Companion to the CPU reproduction notebook. For each of the 120 labelled tasks, we build a context packet with four selectors at an 8k-token budget: BM25 top-k, multi-level packing without a graph, connected + multi-level (EpT), and an oracle with the gold functions first. We give Gemma 4 the change request and the packet and ask it to name up to 5 functions to edit, as JSON. We then score the answer against the functions the reference fix edits. No patch is generated and no tests run.

**Inputs to attach:**
1. The competition dataset (*Google - The Gemma 4 Developer Agent Competition*).
2. The wheelhouse dataset `metric/gemma-4-developer-agent-wheelhouse`, the same one the official starter uses.
3. The model *google/gemma-4*, variant `gemma-4-e4b-it-qat-w4a16-ct` (and optionally `gemma-4-12b-it-qat-w4a16-ct`).

**Environment.** The wheelhouse is built for Python 3.12. Run this inside a copy of the official starter notebook (*Getting Started - Gemma 4 Developer Agent* → Copy & Edit → File → Import Notebook), which keeps the matching environment and inputs. The first cell stops with instructions if the environment is wrong.

**Settings:** Accelerator GPU L4 x4 or GPU T4 x2 (the notebook adapts), Internet off.

**Leakage boundary.** Prompts contain only the problem statement and graph-derived packet text. Gold labels are used only for scoring.

In [ ]:
import glob, importlib, json, logging, os, re, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'

# --- environment checks: stop early with instructions instead of failing deep inside vLLM ---
cands = glob.glob('/kaggle/input/**/gemma-4-developer-agent-wheelhouse', recursive=True)
WHEELHOUSE_DIR = Path(cands[0]) if cands else Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
all_wheels = [w.name for w in WHEELHOUSE_DIR.glob('*.whl')]
assert all_wheels, ('Wheelhouse not attached. Add Input -> Datasets -> "Gemma 4 Developer Agent Wheelhouse" '
                    '(metric/gemma-4-developer-agent-wheelhouse), then run again.')
py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
wheel_tags = sorted(set(re.findall(r'-(cp3\d+)-', ' '.join(all_wheels))))
print('Python', sys.version.split()[0], '| wheelhouse built for:', wheel_tags or 'any')
if wheel_tags and py_tag not in wheel_tags:
    raise SystemExit(
        f'WRONG ENVIRONMENT: this session runs Python {sys.version_info.major}.{sys.version_info.minor} ({py_tag}) '
        f'but the wheelhouse is built for {wheel_tags}. vLLM cannot load here.\n'
        'Fix: open the official starter notebook "Getting Started - Gemma 4 Developer Agent", click Copy & Edit, '
        'then in that copy use File -> Import Notebook to load this file. That copy keeps the right environment.')
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPUs:', r.stdout.strip().replace('\n', ' | ') or 'NONE')
assert r.stdout.strip(), 'No GPU. Settings -> Accelerator -> GPU L4 x4 (or GPU T4 x2), then run again.'

# --- wheelhouse install (from the official starter notebook) ---
shutil.rmtree('/tmp/wheelhouse', ignore_errors=True)
for pth_pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target = tmp_whl / (w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name)
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)
wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

In [ ]:
# --- data and model ---
cands = glob.glob('/kaggle/input/**/tasks.jsonl', recursive=True)
DATA = os.path.dirname(cands[0])
PREFERRED = ['gemma-4-e4b-it-qat-w4a16-ct', 'gemma-4-12b-it-qat-w4a16-ct', 'gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-e2b-it-qat-w4a16-ct']  # first one found is used
configs = [Path(c) for c in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma' in c.lower()]
print('Gemma model folders found:', sorted({str(c.parent) for c in configs}) or 'NONE')
MODEL_NAME = MODEL_PATH = None
for name in PREFERRED:
    hits = sorted(c.parent for c in configs if name in str(c))
    if hits:
        MODEL_NAME, MODEL_PATH = name, hits[-1]
        break
assert MODEL_PATH is not None, 'No Gemma 4 model attached. Add Input -> Models -> google/gemma-4, variant gemma-4-e4b-it-qat-w4a16-ct'
print('data:', DATA, '| model:', MODEL_NAME, '|', MODEL_PATH)

In [ ]:
# --- start vLLM (configuration from the official starter notebook) ---
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

EXTRA = {'gemma-4-12b-it-qat-w4a16-ct': ('--hf-overrides', '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}')}
empty = Path('/tmp/no_adapters'); empty.mkdir(exist_ok=True)
adapters = discover_adapters(str(empty), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()
vllm_cfg = VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1',
    tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=False, max_loras=1, max_lora_rank=128,
    scheduler_reserve_full_isl=False, tensor_parallel_size=tp_size, startup_timeout=60 * 20,
    extra_args=EXTRA.get(MODEL_NAME, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
API = 'http://127.0.0.1:8000/v1'
with urllib.request.urlopen(urllib.request.Request(API + '/models', headers={'Authorization': 'Bearer EMPTY'})) as r:
    SERVED = json.loads(r.read())['data'][0]['id']
PARALLEL = str(4 * tp_size)   # 16 concurrent requests on L4 x4, 8 on T4 x2, 4 on one GPU
print('vLLM up, tp =', tp_size, '| served model id:', SERVED, '| parallel requests:', PARALLEL)

## Source code (same files as the CPU notebook, plus `probe`)

In [ ]:
os.makedirs('evtok', exist_ok=True)

In [ ]:
%%writefile evtok/__init__.py
"""Evidence-per-Token: budgeted, connected, multi-resolution context selection over code graphs.

Licensed under Apache-2.0.
"""

In [ ]:
%%writefile evtok/textproc.py
"""Text utilities: identifier-aware tokenization, query cleaning, token-cost estimates, tier rendering."""
from __future__ import annotations

import re

# Token cost estimate. The Gemma tokenizer is not available offline on the dev machine;
# characters-per-token is calibrated later against the real tokenizer (see README).
CHARS_PER_TOKEN = 3.6

_WORD = re.compile(r"[A-Za-z_][A-Za-z0-9_]*|\d+")
_CAMEL = re.compile(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|\d+")

STOP = set(
    """a an the and or of to in on for with by is are be was were this that it its as at from
    if else elif not no but we you they he she i do does did can could should would will may
    have has had into than then there here which who what when where how why all any some
    self cls none true false return def class import pass raise none ok use using used new
    pr pull request description discussion please link fix fixes add adds added https http www com github
    """.split()
)


def ident_tokens(s: str) -> list[str]:
    """Split text into lowercase tokens; identifiers are also split on snake_case and camelCase.

    The full identifier is kept as an extra token so exact-name matches score higher.
    """
    out: list[str] = []
    for w in _WORD.findall(s):
        lw = w.lower()
        parts = [p.lower() for part in w.split("_") if part for p in _CAMEL.findall(part)]
        if len(parts) > 1 and lw not in STOP:
            out.append(lw)
        out.extend(p for p in parts if p not in STOP and len(p) > 1)
        if len(parts) <= 1 and lw not in STOP and len(lw) > 1 and lw not in parts:
            out.append(lw)
    return out


_HTML_COMMENT = re.compile(r"<!--.*?-->", re.S)
_URL = re.compile(r"https?://\S+")


def clean_problem(ps: str) -> str:
    """Remove PR-template comments and URLs from a problem statement."""
    ps = _HTML_COMMENT.sub(" ", ps)
    ps = _URL.sub(" ", ps)
    return ps


def est_tokens(s: str) -> int:
    return int(len(s) / CHARS_PER_TOKEN) + 1


def signature_text(text: str) -> str:
    """Return the definition header plus the first docstring paragraph (if any)."""
    lines = text.split("\n")
    out: list[str] = []
    i = 0
    # header: decorators + def/class line(s) until a line ending with ':'
    seen_def = False
    while i < len(lines):
        out.append(lines[i])
        seen_def = seen_def or re.match(r"\s*(async\s+def|def|class)\b", lines[i]) is not None
        if seen_def and lines[i].rstrip().endswith(":"):
            i += 1
            break
        i += 1
        if i > 20:
            break
    # docstring
    while i < len(lines) and not lines[i].strip():
        i += 1
    if i < len(lines) and lines[i].strip()[:3] in ('"""', "'''"):
        q = lines[i].strip()[:3]
        doc = [lines[i]]
        if not (lines[i].strip().count(q) >= 2 and len(lines[i].strip()) > 3):
            i += 1
            while i < len(lines) and len(doc) < 8:
                doc.append(lines[i])
                if q in lines[i] or not lines[i].strip():
                    break
                i += 1
        out.extend(doc)
    return "\n".join(out)


def norm_line(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

In [ ]:
%%writefile evtok/data.py
"""Loading tasks, code graphs (node-link JSON) and node embeddings (.npz keyed by node id)."""
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field

import numpy as np

from .textproc import est_tokens, signature_text

TIERS = ("outline", "sig", "full")


def load_tasks(data_dir: str) -> list[dict]:
    with open(os.path.join(data_dir, "tasks.jsonl"), encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def graph_key(task: dict) -> str:
    return task["repo"].split("/")[1] + "_" + task["base_commit"]


def is_test_node(node_id: str) -> bool:
    parts = node_id.split(".")
    return any(p in ("tests", "test", "testing", "conftest") or p.startswith("test_") for p in parts)


@dataclass
class RepoGraph:
    key: str
    ids: list[str]
    text: list[str]
    idx: dict[str, int]
    nbrs: list[np.ndarray]  # undirected neighbours (dedup, no self loops)
    emb: np.ndarray  # (n, d), L2-normalized; zero rows for missing vectors
    has_emb: np.ndarray  # bool (n,)
    cost: np.ndarray  # (n, 3) token cost per tier: outline, sig, full
    pool: np.ndarray  # bool (n,): eligible for context (non-test)
    extra: dict = field(default_factory=dict)
    succ: list = field(default_factory=list)  # directed `calls` successors (used only for packet rendering)


def load_graph(data_dir: str, key: str) -> RepoGraph:
    with open(os.path.join(data_dir, "graphs", key + ".json"), encoding="utf-8") as f:
        g = json.load(f)
    ids = [n["id"] for n in g["nodes"]]
    text = [n.get("text") or "" for n in g["nodes"]]
    idx = {k: i for i, k in enumerate(ids)}
    adj: list[set[int]] = [set() for _ in ids]
    succ: list[set[int]] = [set() for _ in ids]
    for e in g["edges"]:
        s, t = idx.get(e["source"]), idx.get(e["target"])
        if s is None or t is None or s == t:
            continue
        adj[s].add(t)
        adj[t].add(s)
        succ[s].add(t)
    nbrs = [np.fromiter(sorted(a), dtype=np.int64, count=len(a)) for a in adj]

    emb_path = os.path.join(data_dir, "embeddings", key + ".npz")
    d = 256
    emb = np.zeros((len(ids), d), dtype=np.float32)
    has = np.zeros(len(ids), dtype=bool)
    if os.path.exists(emb_path):
        z = np.load(emb_path, allow_pickle=False)
        for k in z.files:
            i = idx.get(k)
            if i is not None:
                v = z[k].astype(np.float32).reshape(-1)
                nv = float(np.linalg.norm(v))
                if nv > 0 and v.shape[0] == d:
                    emb[i] = v / nv
                    has[i] = True

    cost = np.zeros((len(ids), 3), dtype=np.int64)
    for i, (nid, t) in enumerate(zip(ids, text)):
        head = est_tokens(nid) + 3  # "### <id>" header line
        cost[i, 0] = head
        cost[i, 1] = head + est_tokens(signature_text(t))
        cost[i, 2] = head + est_tokens(t)
    cost[:, 1] = np.maximum(cost[:, 1], cost[:, 0] + 1)
    cost[:, 2] = np.maximum(cost[:, 2], cost[:, 1])
    pool = np.array([not is_test_node(n) for n in ids], dtype=bool)
    return RepoGraph(key, ids, text, idx, nbrs, emb, has, cost, pool, succ=succ)

In [ ]:
%%writefile evtok/gold.py
"""Map a reference (gold) patch onto graph nodes. Used ONLY for evaluation labels and the oracle.

Graph nodes carry no file paths or line numbers, so mapping is textual:
  1. removed ('-') lines of a hunk -> nodes whose source contains that line (most specific id);
  2. hunks with no removed lines -> context lines adjacent to the added block;
  3. fallback -> function named in the hunk header (git's nearest preceding def).
Candidates are restricted to nodes whose dotted id matches the patched file's module path when possible.
"""
from __future__ import annotations

import re
from collections import defaultdict

from .data import RepoGraph
from .textproc import norm_line

_HDR = re.compile(r"^@@[^@]*@@\s*(?:async\s+)?(?:def|class)\s+(\w+)")
TRIVIAL = {"return", "pass", "else:", "try:", "finally:", ")", "]", "}", "),", "return None", "continue", "break"}


def parse_patch(patch: str) -> list[dict]:
    """Return [{'path': str, 'new_file': bool, 'hunks': [{'header': str, 'lines': [(op, text)]}]}]."""
    files: list[dict] = []
    cur = None
    hunk = None
    for line in patch.split("\n"):
        if line.startswith("--- "):
            cur = {"path": None, "new_file": line.startswith("--- /dev/null"), "hunks": []}
            files.append(cur)
            hunk = None
        elif line.startswith("+++ ") and cur is not None:
            p = line[4:].strip()
            cur["path"] = p[2:] if p.startswith("b/") else p
        elif line.startswith("@@") and cur is not None:
            hunk = {"header": line, "lines": []}
            cur["hunks"].append(hunk)
            if line.startswith("@@ -0,0 "):
                cur["new_file"] = True
        elif hunk is not None and line[:1] in ("+", "-", " "):
            hunk["lines"].append((line[:1], line[1:]))
    return [f for f in files if f["path"]]


def module_prefixes(path: str) -> list[str]:
    if not path.endswith(".py"):
        return []
    dotted = path[:-3].replace("/", ".")
    out = [dotted]
    parts = dotted.split(".")
    if parts[0] in ("src", "docs_src", "lib"):
        out.append(".".join(parts[1:]))
    if dotted.endswith(".__init__"):
        out = [p[: -len(".__init__")] for p in out]
    return [p for p in out if p]


def nodes_in_file(g: RepoGraph, path: str) -> set[int]:
    prefs = module_prefixes(path)
    is_init = path.endswith("__init__.py")
    out = set()
    for i, nid in enumerate(g.ids):
        for p in prefs:
            if nid == p or nid.startswith(p + "."):
                if is_init:
                    rest = nid[len(p) + 1:]
                    # __init__ members are p.name or p.Class.method; exclude submodules (lowercase.x.y)
                    if rest.count(".") >= 2:
                        continue
                out.add(i)
                break
    return out


def _line_index(g: RepoGraph) -> dict[str, set[int]]:
    if "line_index" not in g.extra:
        idx: dict[str, set[int]] = defaultdict(set)
        for i, t in enumerate(g.text):
            for l in t.split("\n"):
                n = norm_line(l)
                if len(n) >= 8 and n not in TRIVIAL:
                    idx[n].add(i)
        g.extra["line_index"] = idx
    return g.extra["line_index"]


def _most_specific(g: RepoGraph, cands: set[int]) -> set[int]:
    if not cands:
        return set()
    # prefer the deepest ids (method over class over module-level)
    best = max(g.ids[c].count(".") for c in cands)
    deep = {c for c in cands if g.ids[c].count(".") == best}
    # if one candidate's id is a prefix of another, keep the longer
    return {c for c in deep if not any(o != c and g.ids[o].startswith(g.ids[c] + ".") for o in deep)}


def gold_nodes(g: RepoGraph, patch: str) -> dict:
    """Return {'nodes': set[int], 'files': list[str], 'file_nodes': set[int], 'addition_only': bool}."""
    li = _line_index(g)
    gold: set[int] = set()
    file_nodes: set[int] = set()
    files = parse_patch(patch)
    for f in files:
        in_file = nodes_in_file(g, f["path"])
        file_nodes |= in_file
        if f["new_file"]:
            continue
        for h in f["hunks"]:
            hits: set[int] = set()
            lines = h["lines"]
            removed = [t for op, t in lines if op == "-"]
            probe = list(removed)
            if not removed:
                for k, (op, t) in enumerate(lines):
                    if op == "+":
                        if k > 0 and lines[k - 1][0] == " ":
                            probe.append(lines[k - 1][1])
                        if k + 1 < len(lines) and lines[k + 1][0] == " ":
                            probe.append(lines[k + 1][1])
            for t in probe:
                n = norm_line(t)
                if len(n) < 8 or n in TRIVIAL:
                    continue
                c = li.get(n, set())
                if in_file:
                    c = c & in_file
                hits |= _most_specific(g, c)
            if not hits:
                m = _HDR.match(h["header"])
                if m:
                    name = m.group(1)
                    pool = in_file if in_file else range(len(g.ids))
                    c = {i for i in pool if g.ids[i].rsplit(".", 1)[-1] == name}
                    if len(c) <= 3:
                        hits |= c
            gold |= hits
    return {
        "nodes": gold,
        "files": [f["path"] for f in files],
        "file_nodes": file_nodes,
        "addition_only": not gold,
    }

In [ ]:
%%writefile evtok/retrieval.py
"""Leakage-free relevance signals from the problem statement only.

- BM25 over identifier-split node id + source text.
- Identifier anchoring: names mentioned in the PR text resolved to graph nodes
  (same resolution order as the harness: exact -> dotted suffix -> last component).
- Dense pseudo-query: mean embedding of anchor nodes (or of BM25 top-m when no anchor resolves),
  scored by cosine similarity. No text encoder exists for the 256-d node space.
"""
from __future__ import annotations

import math
import re
from collections import Counter

import numpy as np

from .data import RepoGraph
from .textproc import STOP, clean_problem, ident_tokens


class BM25:
    def __init__(self, docs: list[list[str]], k1: float = 1.2, b: float = 0.75):
        self.k1, self.b = k1, b
        self.n = len(docs)
        self.dl = np.array([len(d) for d in docs], dtype=np.float32)
        self.avgdl = float(self.dl.mean()) if self.n else 1.0
        self.tf: list[Counter] = [Counter(d) for d in docs]
        df: Counter = Counter()
        for c in self.tf:
            df.update(c.keys())
        self.idf = {t: math.log(1 + (self.n - v + 0.5) / (v + 0.5)) for t, v in df.items()}
        self.post: dict[str, list[tuple[int, int]]] = {}
        for i, c in enumerate(self.tf):
            for t, v in c.items():
                self.post.setdefault(t, []).append((i, v))

    def score(self, q: list[str]) -> np.ndarray:
        s = np.zeros(self.n, dtype=np.float32)
        norm = self.k1 * (1 - self.b + self.b * self.dl / self.avgdl)
        for t, qtf in Counter(q).items():
            if t not in self.idf:
                continue
            idf = self.idf[t]
            for i, v in self.post[t]:
                s[i] += idf * v * (self.k1 + 1) / (v + norm[i])
        return s


def bm25_for_graph(g: RepoGraph) -> BM25:
    if "bm25" not in g.extra:
        docs = []
        for nid, t in zip(g.ids, g.text):
            idt = ident_tokens(nid.replace(".", " "))
            docs.append(idt + idt + ident_tokens(t[:20000]))
        g.extra["bm25"] = BM25(docs)
    return g.extra["bm25"]


_BT = re.compile(r"`([^`\n]{2,80})`")
_DOTTED = re.compile(r"\b[A-Za-z_]\w*(?:\.[A-Za-z_]\w*)+\b")
_CAMEL = re.compile(r"\b[A-Z][a-z0-9]+(?:[A-Z][a-z0-9]*)+\b")
_SNAKE = re.compile(r"\b[a-z][a-z0-9]*_[a-z0-9_]+\b")
_CALL = re.compile(r"\b([A-Za-z_]\w{2,})\(")


def extract_identifiers(ps: str) -> list[str]:
    ps = clean_problem(ps)
    cands: list[str] = []
    for m in _BT.findall(ps):
        cands.extend(re.findall(r"[A-Za-z_][\w.]*", m))
    cands += _DOTTED.findall(ps) + _CAMEL.findall(ps) + _SNAKE.findall(ps) + _CALL.findall(ps)
    out, seen = [], set()
    for c in cands:
        c = c.strip(".")
        if len(c) < 3 or c.lower() in STOP or c in seen:
            continue
        if re.fullmatch(r"[a-z]+\.(py|md|txt|toml|cfg|yaml|yml|json|html)", c):
            continue
        seen.add(c)
        out.append(c)
    return out


def resolve_anchors(g: RepoGraph, names: list[str], max_matches: int = 8) -> dict[int, float]:
    if "by_last" not in g.extra:
        by_last: dict[str, list[int]] = {}
        for i, nid in enumerate(g.ids):
            by_last.setdefault(nid.rsplit(".", 1)[-1], []).append(i)
        g.extra["by_last"] = by_last
    by_last = g.extra["by_last"]
    w: dict[int, float] = {}
    for c in names:
        if c.endswith(".py"):
            c = c[:-3].replace("/", ".")
        m: list[int] = []
        if c in g.idx:
            m = [g.idx[c]]
        else:
            last = c.rsplit(".", 1)[-1]
            cand = by_last.get(last, [])
            if "." in c:
                m = [i for i in cand if g.ids[i].endswith("." + c)]
            if not m:
                m = list(cand)
            if not m and "." in c:
                # module path mention, e.g. fastapi.routing -> all nodes under it are too many; skip
                m = []
        m = [i for i in m if g.pool[i]]
        if 0 < len(m) <= max_matches:
            for i in m:
                w[i] = max(w.get(i, 0.0), 1.0 / len(m))
    return w


def zscore(x: np.ndarray, mask: np.ndarray) -> np.ndarray:
    v = x[mask]
    sd = float(v.std()) if v.size else 0.0
    z = (x - float(v.mean() if v.size else 0.0)) / (sd if sd > 1e-9 else 1.0)
    z[~mask] = -1e9
    return z


def task_signals(g: RepoGraph, problem: str, m_centroid: int = 5) -> dict:
    """All per-task relevance signals, computed from the problem statement only."""
    q = ident_tokens(clean_problem(problem))
    bm = bm25_for_graph(g).score(q)
    bm[~g.pool] = 0.0
    names = extract_identifiers(problem)
    anchors = resolve_anchors(g, names)
    anc = np.zeros(len(g.ids), dtype=np.float32)
    for i, w in anchors.items():
        anc[i] = w

    def dense_from(nodes: list[int]) -> np.ndarray:
        nodes = [i for i in nodes if g.has_emb[i]]
        if not nodes:
            return np.zeros(len(g.ids), dtype=np.float32)
        qv = g.emb[nodes].mean(axis=0)
        n = float(np.linalg.norm(qv))
        return (g.emb @ (qv / n)) if n > 0 else np.zeros(len(g.ids), dtype=np.float32)

    order = np.argsort(-bm, kind="stable")
    top_bm = [int(i) for i in order[:m_centroid] if bm[i] > 0]
    dense_bm = dense_from(top_bm)
    dense_anchor = dense_from(list(anchors.keys())) if anchors else dense_bm
    return {
        "bm25": bm,
        "anchor": anc,
        "dense_anchor": dense_anchor,
        "dense_bm": dense_bm,
        "n_anchor": len(anchors),
        "names": names,
    }


def hybrid_score(g: RepoGraph, sig: dict, w_dense: float, w_anchor: float, dense_src: str = "anchor") -> np.ndarray:
    mask = g.pool
    s = zscore(np.log1p(sig["bm25"]), mask)
    if w_dense:
        d = sig["dense_anchor"] if dense_src == "anchor" else sig["dense_bm"]
        s = s + w_dense * zscore(d, mask)
    if w_anchor:
        s = s + w_anchor * sig["anchor"] * 3.0
    s[~mask] = -1e9
    return s

In [ ]:
%%writefile evtok/selectors.py
"""Context selectors under a hard token budget B.

A packet is a dict {node_index: tier} with tier in {0: outline, 1: signature+docstring, 2: full source}.
Its cost is sum(cost[v, tier]); every selector guarantees cost <= B.

Selectors
- flat_full:        nodes in score order, full source only (top-k retrieval at matched budget).
- greedy_tiers:     greedy on marginal utility per token over (node, tier-upgrade) actions.
                    expand=False -> flat multi-resolution packing (no graph).
                    expand=True  -> seed-rooted connected selection: a node becomes eligible only
                                    after one of its graph neighbours is selected (or it is a seed).
- graph_expand:     h-hop neighbourhood of seeds re-ranked with a proximity bonus, full source only.
- random_connected: random growth from the same seeds (control).
- oracle:           gold nodes first (full), then filled from another packet (diagnostic upper bound).
"""
from __future__ import annotations

import heapq
from collections import deque

import numpy as np

from .data import RepoGraph

TIER_W = (0.1, 0.45, 1.0)


def packet_cost(g: RepoGraph, pk: dict[int, int]) -> int:
    return int(sum(g.cost[v, t] for v, t in pk.items()))


def flat_full(g: RepoGraph, order, B: int, max_scan: int = 3000) -> dict[int, int]:
    pk: dict[int, int] = {}
    used = 0
    for n, v in enumerate(order):
        if n >= max_scan or B - used < 8:
            break
        v = int(v)
        c = int(g.cost[v, 2])
        if used + c <= B:
            pk[v] = 2
            used += c
    return pk


def bfs_dist(g: RepoGraph, seeds: list[int], h: int) -> dict[int, int]:
    dist = {s: 0 for s in seeds}
    dq = deque(seeds)
    while dq:
        v = dq.popleft()
        if dist[v] >= h:
            continue
        for n in g.nbrs[v]:
            n = int(n)
            if n not in dist and g.pool[n]:
                dist[n] = dist[v] + 1
                dq.append(n)
    return dist


def utility(s: np.ndarray, dist: dict[int, int] | None, beta: float, tau: float, temp: float) -> np.ndarray:
    sp = s.astype(np.float64).copy()
    if dist and beta:
        for v, d in dist.items():
            sp[v] += beta * np.exp(-d / tau)
    smax = float(sp[sp > -1e8].max()) if np.any(sp > -1e8) else 0.0
    u = np.exp(np.clip((sp - smax) / temp, -60, 0))
    u[s <= -1e8] = 0.0
    return u


def greedy_tiers(
    g: RepoGraph,
    util: np.ndarray,
    B: int,
    candidates,
    tiers=(0, 1, 2),
    rho: float = 1.0,
    expand: bool = False,
    region: dict[int, int] | None = None,
    wt=TIER_W,
) -> dict[int, int]:
    cur: dict[int, int] = {}
    heap: list = []
    cnt = 0
    pushed: set[int] = set()

    def push(v: int):
        nonlocal cnt
        c0 = cur.get(v, -1)
        w0 = wt[c0] if c0 >= 0 else 0.0
        k0 = int(g.cost[v, c0]) if c0 >= 0 else 0
        for t in tiers:
            if t <= c0:
                continue
            gain = util[v] * (wt[t] - w0)
            dc = int(g.cost[v, t]) - k0
            if gain <= 0 or dc <= 0:
                continue
            heapq.heappush(heap, (-gain / dc**rho, cnt, v, c0, t, dc))
            cnt += 1

    for v in candidates:
        v = int(v)
        if g.pool[v] and v not in pushed:
            pushed.add(v)
            push(v)
    used = 0
    while heap and B - used >= 4:
        _, _, v, c0, t, dc = heapq.heappop(heap)
        if cur.get(v, -1) != c0 or used + dc > B:
            continue
        cur[v] = t
        used += dc
        if c0 == -1 and expand:
            for n in g.nbrs[v]:
                n = int(n)
                if n in pushed or not g.pool[n] or (region is not None and n not in region):
                    continue
                pushed.add(n)
                push(n)
        push(v)
    return cur


def graph_expand(g: RepoGraph, s: np.ndarray, seeds: list[int], h: int, beta: float, tau: float, B: int) -> dict[int, int]:
    dist = bfs_dist(g, seeds, h)
    region = np.fromiter(dist.keys(), dtype=np.int64)
    sp = s[region] + beta * np.exp(-np.fromiter(dist.values(), dtype=np.float64) / tau)
    order = region[np.argsort(-sp, kind="stable")]
    return flat_full(g, order, B)


def random_connected(g: RepoGraph, seeds: list[int], B: int, rng: np.random.Generator) -> dict[int, int]:
    pk: dict[int, int] = {}
    used = 0
    frontier = list(dict.fromkeys(seeds))
    seen = set(frontier)
    fails = 0
    while frontier and B - used >= 8 and fails < 200:
        v = frontier.pop(int(rng.integers(len(frontier))))
        c = int(g.cost[v, 2])
        if used + c > B:
            fails += 1
            continue
        pk[v] = 2
        used += c
        for n in g.nbrs[v]:
            n = int(n)
            if n not in seen and g.pool[n]:
                seen.add(n)
                frontier.append(n)
    return pk


def oracle(g: RepoGraph, gold: set[int], fill: dict[int, int], B: int) -> dict[int, int]:
    pk: dict[int, int] = {}
    used = 0
    for v in sorted(gold, key=lambda x: int(g.cost[x, 2])):
        c = int(g.cost[v, 2])
        if used + c <= B:
            pk[v] = 2
            used += c
    for v, t in fill.items():
        if v in pk:
            continue
        for tt in range(t, -1, -1):
            c = int(g.cost[v, tt])
            if used + c <= B:
                pk[v] = tt
                used += c
                break
    return pk

In [ ]:
%%writefile evtok/metrics.py
"""Localization metrics for a packet against gold nodes (evaluation only)."""
from __future__ import annotations

import numpy as np

from .data import RepoGraph


def packet_metrics(g: RepoGraph, pk: dict[int, int], gold: dict) -> dict:
    G = gold["nodes"]
    full = {v for v, t in pk.items() if t == 2}
    anyt = set(pk)
    cost = sum(int(g.cost[v, t]) for v, t in pk.items())
    m = {
        "cost": cost,
        "n_nodes": len(pk),
        "n_full": len(full),
        "file_hit": float(bool(full & gold["file_nodes"])) if gold["file_nodes"] else float("nan"),
        "file_hit_any": float(bool(anyt & gold["file_nodes"])) if gold["file_nodes"] else float("nan"),
    }
    if G:
        hit_full = G & full
        m["recall"] = len(hit_full) / len(G)
        m["recall_any"] = len(G & anyt) / len(G)
        m["hit"] = float(bool(hit_full))
        m["hit_any"] = float(bool(G & anyt))
        m["density"] = (sum(int(g.cost[v, 2]) for v in hit_full) / cost) if cost else 0.0
    else:
        for k in ("recall", "recall_any", "hit", "hit_any", "density"):
            m[k] = float("nan")
    # connectivity diagnostics
    sel = set(pk)
    comps = 0
    seen: set[int] = set()
    for v in sel:
        if v in seen:
            continue
        comps += 1
        stack = [v]
        seen.add(v)
        while stack:
            x = stack.pop()
            for n in g.nbrs[x]:
                n = int(n)
                if n in sel and n not in seen:
                    seen.add(n)
                    stack.append(n)
    m["components"] = comps
    return m


def recall_auc(values_by_budget: list[float]) -> float:
    """Mean over log-spaced budgets (equal weight per doubling)."""
    v = [x for x in values_by_budget if not np.isnan(x)]
    return float(np.mean(v)) if v else float("nan")

In [ ]:
%%writefile evtok/run_tier_a.py
"""Tier A: evaluate every selector configuration on every task at every budget (CPU only).

Usage:
    python -m evtok.run_tier_a --data <competition data dir> --out results/tier_a [--workers 4]

Writes <out>/grid.pkl: {instance_id: {"meta": {...}, "res": {config_key: {budget: metrics}}}}.
Hyper-parameter selection (5-fold CV) and statistics are done afterwards by evtok.analyze;
nothing here looks at gold labels except to compute metrics after a packet is built.
"""
from __future__ import annotations

import argparse
import itertools
import os
import pickle
import time
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np

from . import selectors as S
from .data import graph_key, load_graph, load_tasks
from .gold import gold_nodes
from .metrics import packet_metrics
from .retrieval import hybrid_score, task_signals, zscore

BUDGETS = (2000, 4000, 8000, 16000)
KEEP = ("recall", "recall_any", "hit", "hit_any", "density", "file_hit", "file_hit_any", "cost", "n_nodes", "n_full", "components")

HYBRIDS = [(0.0, 0.0, "anchor")] + [
    (wd, wa, src) for wd in (0.0, 0.5, 1.0) for wa in (0.0, 1.0, 2.0) for src in ("anchor", "bm") if not (wd == 0.0 and src == "bm") and (wd, wa) != (0.0, 0.0)
]
TIERW = {"a": (0.02, 0.2, 1.0), "b": (0.1, 0.45, 1.0)}
CONN_GRID = list(itertools.product((0.0, 1.0), (1, 2), (3, 5, 10), (0.5, 1.0), ("a", "b")))  # beta,h,K,rho,tierw
GX_GRID = list(itertools.product((0.0, 1.0), (1, 2), (3, 5, 10)))  # beta,h,K
FLAT_TIER_GRID = list(itertools.product((0.5, 1.0), ("a", "b")))
TAU, TEMP = 1.0, 1.5


def hkey(h):
    return f"wd{h[0]}_wa{h[1]}_{h[2]}"


def _m(g, pk, gold):
    m = packet_metrics(g, pk, gold)
    return {k: m[k] for k in KEEP}


def run_group(data_dir: str, key: str, tasks: list[dict], seed: int = 0) -> dict:
    g = load_graph(data_dir, key)
    out = {}
    for t in tasks:
        gold = gold_nodes(g, t["patch"])
        sig = task_signals(g, t["problem_statement"])
        res: dict[str, dict[int, dict]] = defaultdict(dict)
        # pure dense baselines
        for name, d in (("dense_anchor", sig["dense_anchor"]), ("dense_bm", sig["dense_bm"])):
            s = zscore(d.astype(np.float64), g.pool)
            order = np.argsort(-s, kind="stable")
            for B in BUDGETS:
                res[f"flat|{name}"][B] = _m(g, S.flat_full(g, order, B), gold)
        rng = np.random.default_rng(seed)
        for h in HYBRIDS:
            hk = hkey(h)
            s = hybrid_score(g, sig, *h)
            order = np.argsort(-s, kind="stable")
            top = [int(v) for v in order[:400] if s[v] > -1e8]
            seeds_by_k = {k: top[:k] for k in (3, 5, 10)}
            u_flat = S.utility(s, None, 0.0, TAU, TEMP)
            dists = {(k, hh): S.bfs_dist(g, seeds_by_k[k], hh) for k in (3, 5, 10) for hh in (1, 2)}
            utils = {(k, hh, beta): S.utility(s, dists[(k, hh)], beta, TAU, TEMP) for k in (3, 5, 10) for hh in (1, 2) for beta in (0.0, 1.0)}
            for B in BUDGETS:
                ff = S.flat_full(g, order, B)
                res[f"flat|{hk}"][B] = _m(g, ff, gold)
                for rho, tw in FLAT_TIER_GRID:
                    pk = S.greedy_tiers(g, u_flat, B, top, rho=rho, expand=False, wt=TIERW[tw])
                    res[f"flatT|{hk}|rho{rho}_{tw}"][B] = _m(g, pk, gold)
                for beta, hh, k in GX_GRID:
                    pk = S.graph_expand(g, s, seeds_by_k[k], hh, beta, TAU, B)
                    res[f"gx|{hk}|b{beta}_h{hh}_k{k}"][B] = _m(g, pk, gold)
                    pk = S.greedy_tiers(g, utils[(k, hh, beta)], B, seeds_by_k[k], tiers=(2,), rho=1.0, expand=True, region=dists[(k, hh)])
                    res[f"conn|{hk}|b{beta}_h{hh}_k{k}"][B] = _m(g, pk, gold)
                for beta, hh, k, rho, tw in CONN_GRID:
                    pk = S.greedy_tiers(g, utils[(k, hh, beta)], B, seeds_by_k[k], rho=rho, expand=True, region=dists[(k, hh)], wt=TIERW[tw])
                    res[f"connT|{hk}|b{beta}_h{hh}_k{k}_rho{rho}_{tw}"][B] = _m(g, pk, gold)
                # random connected control (3 draws, averaged later), seeds = top-5
                for r in range(3):
                    pk = S.random_connected(g, seeds_by_k[5], B, rng)
                    res[f"rand|{hk}|r{r}"][B] = _m(g, pk, gold)
                res[f"oracle|{hk}"][B] = _m(g, S.oracle(g, gold["nodes"], ff, B), gold)
        meta = {
            "repo": t["repo"],
            "graph": key,
            "n_gold": len(gold["nodes"]),
            "gold_ids": sorted(g.ids[v] for v in gold["nodes"]),
            "gold_in_pool": all(g.pool[v] for v in gold["nodes"]),
            "addition_only": gold["addition_only"],
            "n_files": len(gold["files"]),
            "n_anchor": sig["n_anchor"],
            "n_nodes_graph": len(g.ids),
            "n_pool": int(g.pool.sum()),
            "gold_cost_full": int(sum(g.cost[v, 2] for v in gold["nodes"])),
        }
        out[t["instance_id"]] = {"meta": meta, "res": dict(res)}
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="results/tier_a")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--limit", type=int, default=0, help="debug: only first N graph groups")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    tasks = load_tasks(a.data)
    groups: dict[str, list[dict]] = defaultdict(list)
    for t in tasks:
        groups[graph_key(t)].append(t)
    keys = sorted(groups)
    if a.limit:
        keys = keys[: a.limit]
    t0 = time.time()
    allres: dict = {}
    with ProcessPoolExecutor(max_workers=a.workers) as ex:
        futs = {ex.submit(run_group, a.data, k, groups[k]): k for k in keys}
        for n, f in enumerate(as_completed(futs), 1):
            allres.update(f.result())
            print(f"[{n}/{len(keys)}] {futs[f]}  {time.time() - t0:.0f}s", flush=True)
    with open(os.path.join(a.out, "grid.pkl"), "wb") as f:
        pickle.dump({"budgets": BUDGETS, "results": allres}, f)
    print("done", len(allres), "tasks", f"{time.time() - t0:.0f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evtok/probe.py
"""Tier A.5: does Gemma 4 use the evidence in a packet? A localization probe.

For each task and packet method, render the packet as text, ask the model to name the functions
that must be edited (JSON), and score the answer against gold nodes.

Usage (on a machine with an OpenAI-compatible server, e.g. vLLM on 127.0.0.1:8000):
    python -m evtok.probe --data <DATA_DIR> --out results/probe --api http://127.0.0.1:8000/v1 --model <served name>
    python -m evtok.probe ... --mock        # offline smoke test, no model

Leakage boundary: prompts contain only the problem statement and packet text built from graphs and
embeddings. Gold nodes are used only to score answers.
"""
from __future__ import annotations

import argparse
import csv
import json
import os
import re
import time
import threading
import urllib.error
import urllib.request
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np

from . import selectors as S
from .data import graph_key, load_graph, load_tasks
from .gold import gold_nodes
from .retrieval import hybrid_score, task_signals
from .run_tier_a import TAU, TEMP, TIERW
from .textproc import clean_problem, signature_text

# Fixed configurations: the settings chosen most often by Tier A cross-validation.
METHODS = {
    "BM25 top-k": dict(kind="flat", h=(0.0, 0.0, "anchor")),
    "Hybrid + tiers (no graph)": dict(kind="flatT", h=(0.5, 1.0, "bm"), rho=0.5, tw="a"),
    "Connected + tiers (EpT)": dict(kind="connT", h=(0.0, 0.0, "anchor"), beta=1.0, hh=2, k=10, rho=0.5, tw="a"),
    "Oracle (gold first)": dict(kind="oracle", h=(0.0, 0.0, "anchor")),
    # Graph-serialization ablation: one EpT packet built at 0.9B, rendered three ways.
    "EpT 0.9B, no graph text": dict(kind="connT", h=(0.0, 0.0, "anchor"), beta=1.0, hh=2, k=10, rho=0.5, tw="a", scale=0.9, enc="none"),
    "EpT 0.9B + edge list": dict(kind="connT", h=(0.0, 0.0, "anchor"), beta=1.0, hh=2, k=10, rho=0.5, tw="a", scale=0.9, enc="edges"),
    "EpT 0.9B + per-entry adjacency": dict(kind="connT", h=(0.0, 0.0, "anchor"), beta=1.0, hh=2, k=10, rho=0.5, tw="a", scale=0.9, enc="adj"),
}
DEFAULT_METHODS = ["BM25 top-k", "Hybrid + tiers (no graph)", "Connected + tiers (EpT)", "Oracle (gold first)"]

SYSTEM = (
    "You localize code changes. You are given a change request for a Python repository and a context "
    "packet: functions and classes from the repository, each under a header line '### <dotted.id>'. "
    "Some entries show full source, some only a signature, some only the id. "
    "Name the functions or classes that must be edited to implement the request, most likely first, "
    "at most 5, using ids exactly as written in the packet. "
    'Reply with JSON only: {"functions": ["dotted.id", ...]}'
)


def build_packet(g, sig, gold, method: str, B: int) -> dict[int, int]:
    cfg = METHODS[method]
    B = int(B * cfg.get("scale", 1.0))
    s = hybrid_score(g, sig, *cfg["h"])
    order = np.argsort(-s, kind="stable")
    if cfg["kind"] == "flat":
        return S.flat_full(g, order, B)
    if cfg["kind"] == "oracle":
        return S.oracle(g, gold["nodes"], S.flat_full(g, order, B), B)
    top = [int(v) for v in order[:400] if s[v] > -1e8]
    if cfg["kind"] == "flatT":
        return S.greedy_tiers(g, S.utility(s, None, 0.0, TAU, TEMP), B, top, rho=cfg["rho"], expand=False, wt=TIERW[cfg["tw"]])
    seeds = top[: cfg["k"]]
    dist = S.bfs_dist(g, seeds, cfg["hh"])
    u = S.utility(s, dist, cfg["beta"], TAU, TEMP)
    return S.greedy_tiers(g, u, B, seeds, rho=cfg["rho"], expand=True, region=dist, wt=TIERW[cfg["tw"]])


def render(g, pk: dict[int, int], s_order: dict[int, int], enc: str = "none") -> str:
    """Full-source entries first, then signatures, then ids; each group in relevance order.

    enc: "none" (no graph text), "edges" (a call-edge list among packet entries at the end),
    "adj" (a 'calls: ... | called by: ...' line under each entry header).
    """
    sel = set(pk)
    calls = {v: [w for w in sorted(g.succ[v]) if w in sel] for v in sel} if g.succ else {v: [] for v in sel}
    called_by: dict[int, list[int]] = {v: [] for v in sel}
    for v, ws in calls.items():
        for w in ws:
            called_by[w].append(v)
    parts = []
    for tier in (2, 1, 0):
        nodes = sorted((v for v, t in pk.items() if t == tier), key=lambda v: s_order.get(v, 10**9))
        for v in nodes:
            head = f"### {g.ids[v]}" + {2: "", 1: " (signature only)", 0: " (id only)"}[tier]
            if enc == "adj" and (calls[v] or called_by[v]):
                head += "\ncalls: " + (", ".join(g.ids[w] for w in calls[v]) or "-") + " | called by: " + (", ".join(g.ids[w] for w in called_by[v]) or "-")
            if tier == 2:
                parts.append(f"{head}\n```python\n{g.text[v]}\n```")
            elif tier == 1:
                parts.append(f"{head}\n```python\n{signature_text(g.text[v])}\n```")
            else:
                parts.append(head)
    if enc == "edges":
        edges = [f"- {g.ids[v]} -> {g.ids[w]}" for v in sorted(sel, key=lambda x: s_order.get(x, 10**9)) for w in calls[v]]
        parts.append("## Call edges among packet entries (caller -> callee)\n" + ("\n".join(edges) if edges else "(none)"))
    return "\n\n".join(parts)


def make_prompt(task: dict, packet_text: str) -> list[dict]:
    user = (
        f"Repository: {task['repo']}\n\n## Change request\n{clean_problem(task['problem_statement']).strip()[:6000]}\n\n"
        f"## Context packet\n{packet_text}\n\nReply with JSON only."
    )
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


def parse_answer(text: str) -> list[str]:
    text = re.sub(r"<\|channel>.*?<channel\|>", "", text or "", flags=re.S)
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            v = json.loads(m.group(0)).get("functions", [])
            return [str(x).strip().strip("`") for x in v if isinstance(x, (str, int))][:5]
        except (json.JSONDecodeError, AttributeError):
            pass
    return re.findall(r"[A-Za-z_][\w]*(?:\.[A-Za-z_][\w]*)+", text)[:5]


def resolve(g, names: list[str]) -> list[int]:
    out = []
    for n in names:
        n = n.split("(")[0].strip()
        if n in g.idx:
            out.append(g.idx[n])
            continue
        c = [i for i, nid in enumerate(g.ids) if nid.endswith("." + n)]
        if len(c) == 1:
            out.append(c[0])
    return out


def call_openai(api: str, model: str, messages: list[dict], max_tokens: int, timeout: int = 600, no_think_kwarg: bool = True) -> dict:
    body = {"model": model, "messages": messages, "temperature": 0.0, "max_tokens": max_tokens, "seed": 0}
    if no_think_kwarg:
        body["chat_template_kwargs"] = {"enable_thinking": False}
    req = urllib.request.Request(
        api.rstrip("/") + "/chat/completions",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json", "Authorization": "Bearer " + os.environ.get("LOCAL_API_KEY", "EMPTY")},
    )
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            resp = json.loads(r.read())
    except urllib.error.HTTPError as e:  # surface the server's message, not just the status code
        detail = e.read().decode("utf-8", "replace")[:500]
        if e.code == 400 and no_think_kwarg and "chat_template_kwargs" in detail:
            return call_openai(api, model, messages, max_tokens, timeout, no_think_kwarg=False)
        raise RuntimeError(f"HTTP {e.code}: {detail}") from None
    msg = resp["choices"][0]["message"]
    return {
        # some servers put the answer in reasoning_content when a reasoning parser is active
        "text": msg.get("content") or msg.get("reasoning_content") or msg.get("reasoning") or "",
        "prompt_tokens": resp.get("usage", {}).get("prompt_tokens"),
        "completion_tokens": resp.get("usage", {}).get("completion_tokens"),
        "latency": time.time() - t0,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="results/probe")
    ap.add_argument("--api", default="http://127.0.0.1:8000/v1")
    ap.add_argument("--model", default="")
    ap.add_argument("--budget", type=int, default=8000)
    ap.add_argument("--methods", default=",".join(DEFAULT_METHODS), help="comma-separated; 'all' runs every method incl. the serialization ablation")
    ap.add_argument("--parallel", type=int, default=16)
    ap.add_argument("--max-tokens", type=int, default=512)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--mock", action="store_true", help="no model: answer with the first full-source id")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    if a.methods == "all":
        methods = list(METHODS)
    elif a.methods == "ablation":  # the three graph-serialization renderings (their names contain commas)
        methods = [m for m in METHODS if m not in DEFAULT_METHODS]
    else:
        methods = [m for m in a.methods.split(",") if m]
    tasks = load_tasks(a.data)
    if a.limit:
        tasks = tasks[: a.limit]
    groups = defaultdict(list)
    for t in tasks:
        groups[graph_key(t)].append(t)

    jobs = []  # (task, method, messages, meta)
    graphs = {}
    for key in sorted(groups):
        g = load_graph(a.data, key)
        for t in groups[key]:
            gold = gold_nodes(g, t["patch"])
            if not gold["nodes"]:
                continue
            sig = task_signals(g, t["problem_statement"])
            order = np.argsort(-hybrid_score(g, sig, 0.0, 0.0, "anchor"), kind="stable")
            rank = {int(v): r for r, v in enumerate(order[:5000])}
            for m in methods:
                pk = build_packet(g, sig, gold, m, a.budget)
                text = render(g, pk, rank, METHODS[m].get("enc", "none"))
                full = {v for v, tt in pk.items() if tt == 2}
                jobs.append((t, m, make_prompt(t, text), {
                    "graph": key, "gold": sorted(gold["nodes"]), "in_full": bool(full & gold["nodes"]),
                    "in_any": bool(set(pk) & gold["nodes"]), "est_cost": S.packet_cost(g, pk), "chars": len(text),
                    "first_full": g.ids[min(full, key=lambda v: rank.get(v, 10**9))] if full else "",
                }))
        g.extra.clear()  # drop BM25 index and line index; keep ids/text for scoring
        graphs[key] = g
    print(f"{len(jobs)} prompts built", flush=True)

    if not a.mock:  # fail fast with a readable message if the server or model name is wrong
        try:
            pre = call_openai(a.api, a.model, jobs[0][2], a.max_tokens, timeout=900)
        except Exception as e:  # noqa: BLE001
            raise SystemExit(f"PREFLIGHT FAILED, nothing was run: {e!r}")
        print(f"preflight ok: {pre['prompt_tokens']} prompt tokens, {pre['latency']:.1f}s, answer: {pre['text'][:200]!r}", flush=True)

    done, lock = [0], threading.Lock()

    def run(job):
        t, m, msgs, meta = job
        if a.mock:
            return {"text": json.dumps({"functions": [meta["first_full"]]}), "prompt_tokens": None, "completion_tokens": None, "latency": 0.0}
        err, out = "", None
        for attempt in range(3):
            try:
                out = call_openai(a.api, a.model, msgs, a.max_tokens)
                break
            except Exception as e:  # noqa: BLE001
                err = repr(e)
                time.sleep(5 * (attempt + 1))
        with lock:
            done[0] += 1
            if done[0] % 50 == 0:
                print(f"  {done[0]}/{len(jobs)} responses, {time.time() - t0:.0f}s", flush=True)
        return out or {"text": "", "error": err, "prompt_tokens": None, "completion_tokens": None, "latency": None}

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=1 if a.mock else a.parallel) as ex:
        outs = list(ex.map(run, jobs))
    print(f"generation done in {time.time() - t0:.0f}s", flush=True)

    # Real tokenizer cost of the packet alone: prompt tokens minus the same prompt with an empty packet.
    base_tok = {}
    if not a.mock:
        seen = {}
        for t, m, msgs, meta in jobs:
            seen.setdefault(t["instance_id"], t)

        def base(t):
            try:
                return t["instance_id"], call_openai(a.api, a.model, make_prompt(t, ""), 1)["prompt_tokens"]
            except Exception:  # noqa: BLE001
                return t["instance_id"], None

        with ThreadPoolExecutor(max_workers=a.parallel) as ex:
            base_tok = dict(ex.map(base, seen.values()))
        print(f"baseline prompts done in {time.time() - t0:.0f}s", flush=True)

    rows = []
    for (t, m, msgs, meta), o in zip(jobs, outs):
        g = graphs[meta["graph"]]
        pred = resolve(g, parse_answer(o["text"]))
        gold = set(meta["gold"])
        rows.append({
            "instance_id": t["instance_id"], "repo": t["repo"], "method": m, "n_gold": len(gold),
            "gold_in_full": int(meta["in_full"]), "gold_in_any": int(meta["in_any"]),
            "acc1": int(bool(pred) and pred[0] in gold), "acc5": int(bool(set(pred[:5]) & gold)),
            "n_pred": len(pred), "parse_ok": int(bool(pred)),
            "est_cost": meta["est_cost"], "chars": meta["chars"],
            "prompt_tokens": o.get("prompt_tokens"), "completion_tokens": o.get("completion_tokens"),
            "packet_tokens": (o["prompt_tokens"] - base_tok[t["instance_id"]]) if (o.get("prompt_tokens") and base_tok.get(t["instance_id"])) else None,
            "latency": o.get("latency"), "error": o.get("error", ""),
            "answer": (o["text"] or "")[:400].replace("\n", " "),
        })
    with open(os.path.join(a.out, "probe_per_task.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)

    # summary + waterfall
    lines = [f"# Tier A.5 localization probe (budget {a.budget}, model {a.model or 'MOCK'})\n",
             "| Method | n | Gold in packet (full) | Gold in packet (any tier) | Acc@1 | Acc@5 | Acc@5 given gold in full | Parse OK |",
             "|---|---|---|---|---|---|---|---|"]
    for m in methods:
        r = [x for x in rows if x["method"] == m]
        gf = [x for x in r if x["gold_in_full"]]
        lines.append(f"| {m} | {len(r)} | {np.mean([x['gold_in_full'] for x in r]):.3f} | {np.mean([x['gold_in_any'] for x in r]):.3f} | "
                     f"{np.mean([x['acc1'] for x in r]):.3f} | {np.mean([x['acc5'] for x in r]):.3f} | "
                     f"{(np.mean([x['acc5'] for x in gf]) if gf else float('nan')):.3f} | {np.mean([x['parse_ok'] for x in r]):.3f} |")
    pt = [(x["est_cost"], x["packet_tokens"]) for x in rows if x["packet_tokens"] and x["est_cost"]]
    if pt:
        est, real = np.array(pt, dtype=float).T
        ratio = real / est
        lines.append(f"\nToken calibration (Gemma tokenizer, packet only; n={len(pt)}): real / estimated tokens: median {np.median(ratio):.3f}, "
                     f"10th-90th percentile {np.percentile(ratio, 10):.3f}-{np.percentile(ratio, 90):.3f}. "
                     f"Median real packet size {np.median(real):.0f} tokens for a budget of {a.budget} estimated tokens; "
                     f"{np.mean(real > a.budget):.1%} of packets exceed the budget in real tokens. "
                     "Rendering adds code fences and tier labels that the estimate does not count.")
    lat = [x["latency"] for x in rows if x["latency"]]
    if lat:
        ct = [x["completion_tokens"] for x in rows if x["completion_tokens"]]
        lines.append(f"\nLatency per request (parallel {a.parallel}): median {np.median(lat):.1f}s, 90th percentile {np.percentile(lat, 90):.1f}s; "
                     f"median completion {np.median(ct):.0f} tokens. Requests with errors: {sum(bool(x['error']) for x in rows)} of {len(rows)}.")
    lines.append("\n## Waterfall (share of tasks)\n")
    lines.append("| Method | missed by retrieval | in packet, model misses | model hits (Acc@5) |")
    lines.append("|---|---|---|---|")
    for m in methods:
        r = [x for x in rows if x["method"] == m]
        miss = np.mean([1 - x["gold_in_any"] for x in r])
        used = np.mean([x["gold_in_any"] * (1 - x["acc5"]) for x in r])
        hit = np.mean([x["gold_in_any"] * x["acc5"] for x in r])
        lines.append(f"| {m} | {miss:.3f} | {used:.3f} | {hit:.3f} |")
    open(os.path.join(a.out, "probe_summary.md"), "w", encoding="utf-8").write("\n".join(lines) + "\n")
    print("\n".join(lines))


if __name__ == "__main__":
    main()

## Smoke test (6 tasks, one method)

Stops the notebook with a readable error if the model does not answer, so the long run below never starts on a broken setup.

In [ ]:
def run(*args):
    # Run an evtok module and stream its output live, so long steps show progress.
    t0 = time.time()
    print('running:', ' '.join(args), flush=True)
    p = subprocess.Popen([sys.executable, '-u', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    print(f'[exit code {rc}, {time.time() - t0:.0f}s]', flush=True)
    if rc != 0:
        raise RuntimeError(f'step failed with exit code {rc}; see the output above')

run('-m', 'evtok.probe', '--data', DATA, '--out', 'results/probe_smoke', '--api', API, '--model', SERVED,
    '--limit', '6', '--methods', 'BM25 top-k', '--parallel', '4')
import csv
smoke = list(csv.DictReader(open('results/probe_smoke/probe_per_task.csv', encoding='utf-8')))
for row in smoke:
    print(row['instance_id'], 'acc5', row['acc5'], 'tokens', row['prompt_tokens'], row['error'][:200], '|', row['answer'][:200])
assert not any(row['error'] for row in smoke), 'the model returned errors; fix this before the full run'
assert any(row['parse_ok'] == '1' for row in smoke), 'no answer could be parsed as function ids; check the answers printed above'

## Main probe: 120 tasks × 4 packets at 8k

BM25 top-k, tiers without graph, EpT, and the oracle. On T4 x2 this is the slow step (it can take over an hour); progress prints every 50 responses.

In [ ]:
OUT = f'results/probe_{MODEL_NAME}'
run('-m', 'evtok.probe', '--data', DATA, '--out', OUT, '--api', API, '--model', SERVED, '--parallel', PARALLEL)

## Graph-serialization ablation: 120 tasks × 3 renderings

One EpT packet built at 0.9 × budget and rendered with no graph text, with a call-edge list, or with per-entry `calls / called by` lines. A failure here does not discard the main probe above.

In [ ]:
OUT_ABL = f'results/probe_serialization_{MODEL_NAME}'
ABLATION_OK = True
try:
    run('-m', 'evtok.probe', '--data', DATA, '--out', OUT_ABL, '--api', API, '--model', SERVED, '--parallel', PARALLEL,
        '--methods', 'ablation')
except Exception as e:
    ABLATION_OK = False
    print('serialization ablation did not finish:', e)

## No-LLM baseline: answer with the top-ranked full-source function

In [ ]:
run('-m', 'evtok.probe', '--data', DATA, '--out', 'results/probe_rank1', '--mock')

## Paired comparisons (bootstrap 95% CI, exact McNemar on Acc@5)

In [ ]:
import csv, math
import numpy as np
from collections import defaultdict
rows = list(csv.DictReader(open(f'{OUT}/probe_per_task.csv', encoding='utf-8')))
if ABLATION_OK:
    rows += list(csv.DictReader(open(f'{OUT_ABL}/probe_per_task.csv', encoding='utf-8')))
acc = defaultdict(dict)
for r_ in rows:
    acc[r_['method']][r_['instance_id']] = int(r_['acc5'])
ids = sorted(acc['BM25 top-k'])
def mcn(b, c):
    n = b + c
    return 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2**n)
rng = np.random.default_rng(0)
def compare(m, base):
    d = np.array([acc[m][i] - acc[base][i] for i in ids], dtype=float)
    bs = d[rng.integers(0, len(d), (10000, len(d)))].mean(1)
    b = int(sum(acc[m][i] == 1 and acc[base][i] == 0 for i in ids)); c = int(sum(acc[m][i] == 0 and acc[base][i] == 1 for i in ids))
    print(f'{m} - {base}: dAcc@5 {d.mean():+.3f} [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]  wins/losses {b}/{c}  McNemar p={mcn(b, c):.3f}')
for m in ['Hybrid + tiers (no graph)', 'Connected + tiers (EpT)', 'Oracle (gold first)']:
    compare(m, 'BM25 top-k')
for m in ['EpT 0.9B + edge list', 'EpT 0.9B + per-entry adjacency']:
    if ABLATION_OK:
        compare(m, 'EpT 0.9B, no graph text')

## Results to copy into the paper

Everything between the two marker lines.

In [ ]:
try:
    server_instance.stop()
except Exception as e:
    print('server stop:', e)
print('=' * 20, 'COPY FROM HERE', '=' * 20)
print('model:', MODEL_NAME, '| GPUs:', gpu_count, '| tensor parallel:', tp_size)
print(open(f'{OUT}/probe_summary.md', encoding='utf-8').read())
print('--- graph-serialization ablation ---')
print(open(f'{OUT_ABL}/probe_summary.md', encoding='utf-8').read() if ABLATION_OK else 'not completed')
print('--- no-LLM rank-1 baseline ---')
print(open('results/probe_rank1/probe_summary.md', encoding='utf-8').read())
print('=' * 20, 'COPY TO HERE', '=' * 20)

## Notes

- Decoding: temperature 0, seed 0, thinking disabled, max 512 output tokens. The same prompt template is used for every method, and only the packet differs.
- `packet_tokens` in `probe_per_task.csv` is the real Gemma tokenizer count of the packet alone (prompt tokens minus the same prompt with an empty packet). The summary reports its ratio to the `len/3.6` estimate used for budgets.
- Fixed selector settings are the ones Tier A cross-validation chose most often. They are listed in `evtok/probe.py`.